# 实验五 · RGB → Gray 灰度转换

**所属**：《并行计算》第三章 · ARM NEON SIMD 编程　|　**难度**：⭐⭐ 进阶　|　**预计时长**：30–40 分钟

> **实验说明**
> 1. 本实验采用分步实现的方式：由 **v1 串行版本**起，依次引入 **v2 NEON 加权规约**与 **v3 循环展开**，每一版本均实际编译并运行，据此观察性能的逐步变化。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖 **ARM(aarch64/arm64) + NEON**；请在华为鲲鹏处理器上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. 三个版本的代码为递增关系：后一版本在前一版本基础上新增一个实现方式，输出表格相应增加一行，便于对照阅读，理解优化的引入过程。
> 6. 本实验与**实验四 白平衡**形成对照：白平衡含规约统计与饱和处理，编译器同样能有效自动向量化，但手写 NEON 仍明显更快；而本实验属于规整的逐像素运算，可观察到自动向量化已足以逼近甚至超过手写 NEON。

## 🎯 学习目标

完成本实验后，学生应能够：

- 理解**加权亮度**（BT.601）转换公式，以及为何不采用三通道简单平均
- 掌握**定点化**：权重乘以 256 得到整数 77 / 150 / 29，运算末尾 `>>8` 归一化
- 掌握**加宽乘加** `vmull_u8` / `vmlal_u8`（u8→u16），理解其防止溢出的作用
- 理解**跨通道规约**：将三个通道加权求和为单个灰度平面
- 通过与自动向量化的对比，理解“规整逐像素”负载为何编译器即可有效处理

## 🗺️ 学习路径

1. **准备阶段**：理解按亮度加权的原因，以及浮点权重的定点化
2. **v1 · 串行基准**：实现 `rgb2gray_serial_no_vec`（关闭向量化，作为基准）与 `rgb2gray_serial`（允许自动向量化）
   → 考察编译器对规整逐像素运算的自动向量化能力
3. **v2 · NEON 加权规约**：新增 `rgb2gray_neon`，采用 `vld3` + 加宽乘加 + 窄化
   → 掌握跨通道加权规约的向量化实现
4. **v3 · 循环展开**：新增 `rgb2gray_neon_unroll`
   → 考察循环展开的进一步收益
5. **可视化与分析**：以 v3 的输出结果绘制加速比柱状图，并与自动向量化对比

## 1. 背景与动机

将彩色图像转换为灰度，不能简单地取 `(R+G+B)/3`——因为**人眼对绿色最敏感、对蓝色最不敏感**。标准做法是按**亮度**加权（BT.601 标准），这一权重实际上即为 YUV 颜色空间中的 **Y（亮度）分量**。

本实验读取交织存储的 RGB 图像，进行一次**跨通道加权求和**，输出单通道灰度平面。它的计算量高于 AXPY，又比白平衡简单，是理解“计算密度如何影响加速比”的适宜案例。

## 2. 算法与公式

浮点形式（BT.601 亮度）：
$$Gray = 0.299R + 0.587G + 0.114B$$

定点化（乘以 256）：$0.299{\times}256{\approx}77,\ 0.587{\times}256{\approx}150,\ 0.114{\times}256{\approx}29$，且 $77+150+29=256$ 恰好归一化：
$$Gray = (77R + 150G + 29B + 128) \gg 8$$
其中 `+128` 用于实现四舍五入（而非直接截断），`>>8` 等价于除以 256。

> 定点化的动机与实验四相同：以整数乘加代替浮点运算，在边缘端设备上更高效，且便于 SIMD 并行。

## 3. 核心 NEON 指令与技巧

```c
uint8x8_t w_r=vdup_n_u8(77), w_g=vdup_n_u8(150), w_b=vdup_n_u8(29);
for (i = 0; i + 16 <= n; i += 16) {
  uint8x16x3_t pix = vld3q_u8(rgb + i*3);              // 解交织 R/G/B
  uint16x8_t lo = vmull_u8(vget_low_u8(pix.val[0]), w_r);   // 77*R
  lo = vmlal_u8(lo, vget_low_u8(pix.val[1]), w_g);          // +150*G
  lo = vmlal_u8(lo, vget_low_u8(pix.val[2]), w_b);          // + 29*B
  uint8x8_t g_lo = vrshrn_n_u16(lo, 8);               // 带舍入右移 8 并窄化回 u8
  // 高 8 像素同理，vcombine 后用 vst1q_u8 写入灰度平面
}
```

- **`vmull_u8` / `vmlal_u8`**：加宽乘 / 加宽乘加（u8×u8→u16），将加权和保存在 16 位中以防溢出
- **为何需要加宽**：$256 \times 255 = 65280$，已超出 u8 范围，必须用 u16 承载
- **`vrshrn_n_u16`**：带舍入的右移 8 位并窄化回 u8（对应公式中的 `+128` 与 `>>8`）
- **输出为单通道**：使用 `vst1q_u8` 写入（而非 `vst3`），因为灰度只有一个平面

## 4. 环境准备

In [ ]:
import platform, subprocess, shutil, sys

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
IS_ARM = platform.machine().lower() in ("aarch64", "arm64", "armv7l", "armv8l")
if not IS_ARM:
    print("\n⚠️  当前不是 ARM 架构，NEON 代码无法在此编译运行。")
elif CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
else:
    print("\n✅ 环境就绪：ARM 架构 + 编译器可用，可以开始实验！")

In [ ]:
import subprocess, platform, re, shutil

MACHINE = platform.machine().lower()


def compile_c(src, out):
    """尝试多组编译参数，返回可执行文件名；失败则打印错误。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    if MACHINE in ("armv7l", "armv8l"):  # 32 位 ARM 需显式开 NEON
        flagsets = ["-O3 -fPIC -mfpu=neon -mfloat-abi=hard -march=armv7-a"]
    else:  # aarch64 / arm64：NEON 默认开启
        flagsets = ["-O3 -fPIC"]
    last = ""
    for fl in flagsets:
        cmd = f"{base} {fl} {src} -o {out} -lm"
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode == 0:
            print("✅ 编译成功：", cmd)
            return out
        last = r.stderr
    print("❌ 编译失败：\n", last)
    return None


def run_bin(out, *args):
    """运行可执行文件并打印其输出。"""
    r = subprocess.run(
        [f"./{out}"] + [str(a) for a in args], capture_output=True, text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
    return r.stdout


def parse_table(text):
    """解析 | 方法 | 耗时 | 加速比 | 校验 | 表格，兼容 '4.21 x' 与 '4.21x'。"""
    rows = []
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 3:
            continue
        name = cells[0]
        if name.lower() in ("method", "方法") or set(name) <= set("-: "):
            continue
        mt = re.search(r"[-+]?\d*\.?\d+", cells[1])
        ms = re.search(r"[-+]?\d*\.?\d+", cells[2])
        if not mt:
            continue
        rows.append(
            {
                "method": name,
                "time": float(mt.group()),
                "speedup": float(ms.group()) if ms else None,
            }
        )
    return rows

In [ ]:
import matplotlib.pyplot as plt


def plot_speedup(rows, title=""):
    rows = [r for r in rows if r["speedup"] is not None]
    if not rows:
        print("未解析到可绘制的加速比。")
        return
    names = [r["method"] for r in rows]
    sp = [r["speedup"] for r in rows]
    best = sp.index(max(sp))
    colors = ["#9aa0a6" if s <= 1.05 else "#295E96" for s in sp]
    colors[best] = "#C7000B"  # 最快版本标红
    plt.figure(figsize=(8, 4))
    bars = plt.bar(names, sp, color=colors)
    plt.axhline(1.0, ls="--", c="gray", lw=1)
    for b, s in zip(bars, sp):
        plt.text(
            b.get_x() + b.get_width() / 2,
            s,
            f"{s:.2f}x",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    plt.ylabel("Speedup (x)")
    plt.title(title)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# 创建源代码目录
!mkdir -p src_rgb2gray

## 5. v1 · 串行基准实现

与前面实验一致，第一个版本包含**两个函数体相同**的串行实现，区别仅在于是否允许编译器自动向量化：

<table>
  <thead>
    <tr>
      <th style="text-align: left;">函数</th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>rgb2gray_serial_no_vec</code></td>
      <td style="text-align: left;">通过 <code>no-tree-vectorize</code> <strong>显式关闭</strong>自动向量化，作为<strong>性能基准</strong>（1.00×）</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>rgb2gray_serial</code></td>
      <td style="text-align: left;">源码相同，但<strong>允许编译器自动向量化</strong></td>
    </tr>
  </tbody>
</table>

### 💡 关注点
RGB→Gray 是规整的逐像素运算。请在运行后观察 `Serial (Auto)` 相对基准的加速比——对这类运算，编译器往往能有效地自动向量化。

### 数据布局
- `rgb`：交织存储的输入图像（`3n` 字节）；`gray`：单通道灰度输出（`n` 字节，非原地）
- 参考结果 `g_ref` 由 `rgb2gray_serial_no_vec` 生成；`check_diff` 容差取 1（允许定点舍入的 ±1 差异）

In [ ]:
%%writefile src_rgb2gray/rgb2gray_v1.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

// Fixed-point BT.601 luminance weights (Q8): 77 + 150 + 29 = 256
// Gray = (77*R + 150*G + 29*B + 128) >> 8
enum { WR = 77, WG = 150, WB = 29 };

// Helper: Get monotonic time in MILLISECONDS (ms)
static double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Helper: Verify results against reference (allow +/-1 rounding difference)
static const char* check_diff(const uint8_t* ref, const uint8_t* test, long n) {
  int max_diff = 0;
  for (long i = 0; i < n; i++) {
    int d = abs((int)ref[i] - (int)test[i]);
    if (d > max_diff) max_diff = d;
  }
  if (max_diff <= 1)
    return "PASS";
  else
    return "FAIL";
}

// Serial version (forced no-vectorization) - Baseline
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void rgb2gray_serial_no_vec(const uint8_t* restrict rgb, uint8_t* restrict gray,
                            long n) {
  for (long i = 0; i < n; i++) {
    uint32_t r = rgb[3 * i + 0];
    uint32_t g = rgb[3 * i + 1];
    uint32_t b = rgb[3 * i + 2];
    gray[i] = (uint8_t)((WR * r + WG * g + WB * b + 128) >> 8);
  }
}

// Serial version (compiler may auto-vectorize)
void rgb2gray_serial(const uint8_t* restrict rgb, uint8_t* restrict gray,
                     long n) {
  for (long i = 0; i < n; i++) {
    uint32_t r = rgb[3 * i + 0];
    uint32_t g = rgb[3 * i + 1];
    uint32_t b = rgb[3 * i + 2];
    gray[i] = (uint8_t)((WR * r + WG * g + WB * b + 128) >> 8);
  }
}

// Benchmark helper: run kernel NTIMES and return the AVERAGE time (ms)
typedef void (*gray_fn)(const uint8_t*, uint8_t*, long);

static double bench(gray_fn fn, const uint8_t* rgb, uint8_t* gray, long n) {
  double start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) fn(rgb, gray, n);
  double end = get_time_ms();
  return (end - start) / NTIMES;
}

int main(int argc, char* argv[]) {
  long n = (argc == 2) ? atol(argv[1]) : (1920 * 1080);  // default 1080p

  printf("=======================================================\n");
  printf(" RGB2Gray v1: Serial Baseline and Auto-Vectorized Serial (BT.601)\n");
  printf(" Gray = (77*R + 150*G + 29*B + 128) >> 8\n");
  printf(" Pixels: %ld   Loops: %d\n", n, NTIMES);
  printf("=======================================================\n");

  // Aligned allocation (16-byte) for SIMD efficiency
  size_t bytes_rgb = ((size_t)n * 3 + 15) & ~(size_t)15;
  size_t bytes_g   = ((size_t)n     + 15) & ~(size_t)15;
  uint8_t* rgb = (uint8_t*)aligned_alloc(16, bytes_rgb);
  uint8_t* g_ref = (uint8_t*)aligned_alloc(16, bytes_g);
  uint8_t* g_opt = (uint8_t*)aligned_alloc(16, bytes_g);
  if (!rgb || !g_ref || !g_opt) {
    printf("Alloc failed\n");
    return 1;
  }

  for (long i = 0; i < n; i++) {
    rgb[3 * i + 0] = (uint8_t)((i * 7) & 0xFF);
    rgb[3 * i + 1] = (uint8_t)((i * 13) & 0xFF);
    rgb[3 * i + 2] = (uint8_t)((i * 29) & 0xFF);
  }

  // Golden reference = Serial (No-Vec)
  rgb2gray_serial_no_vec(rgb, g_ref, n);

  memset(g_opt, 0, bytes_g);
  double t_nv = bench(rgb2gray_serial_no_vec, rgb, g_opt, n);
  memset(g_opt, 0, bytes_g);
  double t_se = bench(rgb2gray_serial, rgb, g_opt, n);
  const char* s_se = check_diff(g_ref, g_opt, n);

  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |   -   |\n", t_nv);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_se, t_nv / t_se,
         s_se);
  printf("-------------------------------------------------\n");

  free(rgb);
  free(g_ref);
  free(g_opt);
  return 0;
}

In [ ]:
BIN = compile_c("src_rgb2gray/rgb2gray_v1.c", "src_rgb2gray/rgb2gray_v1")
out_v1 = run_bin(BIN, 2073600)

## 6. v2 · NEON 加权规约实现

在 v1 的基础上**新增 `rgb2gray_neon` 函数**，实现跨通道加权规约的向量化：

```c
uint8x16x3_t pix = vld3q_u8(rgb + i*3);          // 解交织为 R/G/B
uint16x8_t lo = vmull_u8(vget_low_u8(pix.val[0]), w_r);  // 加宽乘
lo = vmlal_u8(lo, vget_low_u8(pix.val[1]), w_g);         // 加宽乘加
lo = vmlal_u8(lo, vget_low_u8(pix.val[2]), w_b);
uint8x8_t g_lo = vrshrn_n_u16(lo, 8);            // 舍入右移 + 窄化
```

### 🔑 知识点
- **加宽以防溢出**：三通道加权和最大约为 $255\times256$，远超 u8，故须在 u16 中累加
- **一条向量指令处理 16 个像素**：`vld3q_u8` 一次解交织 16 个像素，分低 8、高 8 两组处理
- **跨通道规约**：与实验二 GEMV 的规约不同，此处是**固定 3 个通道的加权和**，规约宽度固定且很小，因此实现简单、无跨迭代依赖
- **输出单通道**：结果用 `vst1q_u8` 写入灰度平面

此版本包含**三行**输出。请将 NEON 的加速比与 `Serial (Auto)` 相比较。

In [ ]:
%%writefile src_rgb2gray/rgb2gray_v2.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

// Fixed-point BT.601 luminance weights (Q8): 77 + 150 + 29 = 256
// Gray = (77*R + 150*G + 29*B + 128) >> 8
enum { WR = 77, WG = 150, WB = 29 };

// Helper: Get monotonic time in MILLISECONDS (ms)
static double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Helper: Verify results against reference (allow +/-1 rounding difference)
static const char* check_diff(const uint8_t* ref, const uint8_t* test, long n) {
  int max_diff = 0;
  for (long i = 0; i < n; i++) {
    int d = abs((int)ref[i] - (int)test[i]);
    if (d > max_diff) max_diff = d;
  }
  if (max_diff <= 1)
    return "PASS";
  else
    return "FAIL";
}

// Serial version (forced no-vectorization) - Baseline
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void rgb2gray_serial_no_vec(const uint8_t* restrict rgb, uint8_t* restrict gray,
                            long n) {
  for (long i = 0; i < n; i++) {
    uint32_t r = rgb[3 * i + 0];
    uint32_t g = rgb[3 * i + 1];
    uint32_t b = rgb[3 * i + 2];
    gray[i] = (uint8_t)((WR * r + WG * g + WB * b + 128) >> 8);
  }
}

// Serial version (compiler may auto-vectorize)
void rgb2gray_serial(const uint8_t* restrict rgb, uint8_t* restrict gray,
                     long n) {
  for (long i = 0; i < n; i++) {
    uint32_t r = rgb[3 * i + 0];
    uint32_t g = rgb[3 * i + 1];
    uint32_t b = rgb[3 * i + 2];
    gray[i] = (uint8_t)((WR * r + WG * g + WB * b + 128) >> 8);
  }
}

// NEON intrinsics (16 pixels per iteration)
void rgb2gray_neon(const uint8_t* restrict rgb, uint8_t* restrict gray,
                   long n) {
  // weights broadcast into 8-lane u8 vectors
  uint8x8_t w_r = vdup_n_u8(WR);
  uint8x8_t w_g = vdup_n_u8(WG);
  uint8x8_t w_b = vdup_n_u8(WB);

  long i = 0;
  for (; i <= n - 16; i += 16) {
    // structured load: de-interleave 16 RGB pixels into 3 planes
    uint8x16x3_t pix = vld3q_u8(rgb + i * 3);

    // low 8 pixels: widen u8->u16 while multiplying (vmull/vmlal) to avoid
    // overflow
    uint16x8_t acc_lo = vmull_u8(vget_low_u8(pix.val[0]), w_r);  // 77*R
    acc_lo = vmlal_u8(acc_lo, vget_low_u8(pix.val[1]), w_g);     // +150*G
    acc_lo = vmlal_u8(acc_lo, vget_low_u8(pix.val[2]), w_b);     // + 29*B

    // high 8 pixels
    uint16x8_t acc_hi = vmull_u8(vget_high_u8(pix.val[0]), w_r);
    acc_hi = vmlal_u8(acc_hi, vget_high_u8(pix.val[1]), w_g);
    acc_hi = vmlal_u8(acc_hi, vget_high_u8(pix.val[2]), w_b);

    // rounding shift-right by 8 and narrow back to u8 (>>8 with round)
    uint8x8_t g_lo = vrshrn_n_u16(acc_lo, 8);
    uint8x8_t g_hi = vrshrn_n_u16(acc_hi, 8);

    vst1q_u8(gray + i, vcombine_u8(g_lo, g_hi));
  }

  // tail
  for (; i < n; i++) {
    uint32_t r = rgb[3 * i + 0], g = rgb[3 * i + 1], b = rgb[3 * i + 2];
    gray[i] = (uint8_t)((WR * r + WG * g + WB * b + 128) >> 8);
  }
}

// Benchmark helper: run kernel NTIMES and return the AVERAGE time (ms)
typedef void (*gray_fn)(const uint8_t*, uint8_t*, long);

static double bench(gray_fn fn, const uint8_t* rgb, uint8_t* gray, long n) {
  double start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) fn(rgb, gray, n);
  double end = get_time_ms();
  return (end - start) / NTIMES;
}

int main(int argc, char* argv[]) {
  long n = (argc == 2) ? atol(argv[1]) : (1920 * 1080);  // default 1080p

  printf("=======================================================\n");
  printf(" RGB2Gray v2: Add NEON Weighted Reduction (BT.601)\n");
  printf(" Gray = (77*R + 150*G + 29*B + 128) >> 8\n");
  printf(" Pixels: %ld   Loops: %d\n", n, NTIMES);
  printf("=======================================================\n");

  // Aligned allocation (16-byte) for SIMD efficiency
  size_t bytes_rgb = ((size_t)n * 3 + 15) & ~(size_t)15;
  size_t bytes_g   = ((size_t)n     + 15) & ~(size_t)15;
  uint8_t* rgb = (uint8_t*)aligned_alloc(16, bytes_rgb);
  uint8_t* g_ref = (uint8_t*)aligned_alloc(16, bytes_g);
  uint8_t* g_opt = (uint8_t*)aligned_alloc(16, bytes_g);
  if (!rgb || !g_ref || !g_opt) {
    printf("Alloc failed\n");
    return 1;
  }

  for (long i = 0; i < n; i++) {
    rgb[3 * i + 0] = (uint8_t)((i * 7) & 0xFF);
    rgb[3 * i + 1] = (uint8_t)((i * 13) & 0xFF);
    rgb[3 * i + 2] = (uint8_t)((i * 29) & 0xFF);
  }

  // Golden reference = Serial (No-Vec)
  rgb2gray_serial_no_vec(rgb, g_ref, n);

  memset(g_opt, 0, bytes_g);
  double t_nv = bench(rgb2gray_serial_no_vec, rgb, g_opt, n);
  memset(g_opt, 0, bytes_g);
  double t_se = bench(rgb2gray_serial, rgb, g_opt, n);
  const char* s_se = check_diff(g_ref, g_opt, n);
  memset(g_opt, 0, bytes_g);
  double t_ne = bench(rgb2gray_neon, rgb, g_opt, n);
  const char* s_ne = check_diff(g_ref, g_opt, n);

  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |   -   |\n", t_nv);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_se, t_nv / t_se,
         s_se);
  printf("| NEON Intrinsic  | %9.3f | %5.2f x |  %-4s |\n", t_ne, t_nv / t_ne,
         s_ne);
  printf("-------------------------------------------------\n");

  free(rgb);
  free(g_ref);
  free(g_opt);
  return 0;
}

In [ ]:
BIN = compile_c("src_rgb2gray/rgb2gray_v2.c", "src_rgb2gray/rgb2gray_v2")
out_v2 = run_bin(BIN, 2073600)

## 7. v3 · 循环展开实现

在 v2 的基础上**新增 `rgb2gray_neon_unroll` 函数**：单次迭代处理 **32 个像素**（两个 16 像素块）。

### 🔑 知识点
- **循环展开**：单次迭代处理更多像素，减少循环控制开销，并使多组加宽乘加与访存操作可以重叠
- **各像素相互独立**：本运算无跨迭代依赖，展开主要用于摊薄循环开销、提高访存与计算的重叠度

> ⚠️ **不要预设展开一定更快**：`-O3` 本身已会对循环做一定展开，而本负载又受访存带宽制约，因此手工展开的余地很小。请以自己机器上的实测为准。

至此，**四个实现**（含基准）全部实现。

In [ ]:
%%writefile src_rgb2gray/rgb2gray_v3.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

// Fixed-point BT.601 luminance weights (Q8): 77 + 150 + 29 = 256
// Gray = (77*R + 150*G + 29*B + 128) >> 8
enum { WR = 77, WG = 150, WB = 29 };

// Helper: Get monotonic time in MILLISECONDS (ms)
static double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Helper: Verify results against reference (allow +/-1 rounding difference)
static const char* check_diff(const uint8_t* ref, const uint8_t* test, long n) {
  int max_diff = 0;
  for (long i = 0; i < n; i++) {
    int d = abs((int)ref[i] - (int)test[i]);
    if (d > max_diff) max_diff = d;
  }
  if (max_diff <= 1)
    return "PASS";
  else
    return "FAIL";
}

// Serial version (forced no-vectorization) - Baseline
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void rgb2gray_serial_no_vec(const uint8_t* restrict rgb, uint8_t* restrict gray,
                            long n) {
  for (long i = 0; i < n; i++) {
    uint32_t r = rgb[3 * i + 0];
    uint32_t g = rgb[3 * i + 1];
    uint32_t b = rgb[3 * i + 2];
    gray[i] = (uint8_t)((WR * r + WG * g + WB * b + 128) >> 8);
  }
}

// Serial version (compiler may auto-vectorize)
void rgb2gray_serial(const uint8_t* restrict rgb, uint8_t* restrict gray,
                     long n) {
  for (long i = 0; i < n; i++) {
    uint32_t r = rgb[3 * i + 0];
    uint32_t g = rgb[3 * i + 1];
    uint32_t b = rgb[3 * i + 2];
    gray[i] = (uint8_t)((WR * r + WG * g + WB * b + 128) >> 8);
  }
}

// NEON intrinsics (16 pixels per iteration)
void rgb2gray_neon(const uint8_t* restrict rgb, uint8_t* restrict gray,
                   long n) {
  // weights broadcast into 8-lane u8 vectors
  uint8x8_t w_r = vdup_n_u8(WR);
  uint8x8_t w_g = vdup_n_u8(WG);
  uint8x8_t w_b = vdup_n_u8(WB);

  long i = 0;
  for (; i <= n - 16; i += 16) {
    // structured load: de-interleave 16 RGB pixels into 3 planes
    uint8x16x3_t pix = vld3q_u8(rgb + i * 3);

    // low 8 pixels: widen u8->u16 while multiplying (vmull/vmlal) to avoid
    // overflow
    uint16x8_t acc_lo = vmull_u8(vget_low_u8(pix.val[0]), w_r);  // 77*R
    acc_lo = vmlal_u8(acc_lo, vget_low_u8(pix.val[1]), w_g);     // +150*G
    acc_lo = vmlal_u8(acc_lo, vget_low_u8(pix.val[2]), w_b);     // + 29*B

    // high 8 pixels
    uint16x8_t acc_hi = vmull_u8(vget_high_u8(pix.val[0]), w_r);
    acc_hi = vmlal_u8(acc_hi, vget_high_u8(pix.val[1]), w_g);
    acc_hi = vmlal_u8(acc_hi, vget_high_u8(pix.val[2]), w_b);

    // rounding shift-right by 8 and narrow back to u8 (>>8 with round)
    uint8x8_t g_lo = vrshrn_n_u16(acc_lo, 8);
    uint8x8_t g_hi = vrshrn_n_u16(acc_hi, 8);

    vst1q_u8(gray + i, vcombine_u8(g_lo, g_hi));
  }

  // tail
  for (; i < n; i++) {
    uint32_t r = rgb[3 * i + 0], g = rgb[3 * i + 1], b = rgb[3 * i + 2];
    gray[i] = (uint8_t)((WR * r + WG * g + WB * b + 128) >> 8);
  }
}

// NEON with loop unrolling (32 pixels per iteration)
void rgb2gray_neon_unroll(const uint8_t* restrict rgb, uint8_t* restrict gray,
                          long n) {
  uint8x8_t w_r = vdup_n_u8(WR);
  uint8x8_t w_g = vdup_n_u8(WG);
  uint8x8_t w_b = vdup_n_u8(WB);

  long i = 0;
  for (; i <= n - 32; i += 32) {
    uint8x16x3_t p0 = vld3q_u8(rgb + i * 3);
    uint8x16x3_t p1 = vld3q_u8(rgb + (i + 16) * 3);

    uint16x8_t a0 = vmull_u8(vget_low_u8(p0.val[0]), w_r);
    a0 = vmlal_u8(a0, vget_low_u8(p0.val[1]), w_g);
    a0 = vmlal_u8(a0, vget_low_u8(p0.val[2]), w_b);
    uint16x8_t a1 = vmull_u8(vget_high_u8(p0.val[0]), w_r);
    a1 = vmlal_u8(a1, vget_high_u8(p0.val[1]), w_g);
    a1 = vmlal_u8(a1, vget_high_u8(p0.val[2]), w_b);
    uint16x8_t a2 = vmull_u8(vget_low_u8(p1.val[0]), w_r);
    a2 = vmlal_u8(a2, vget_low_u8(p1.val[1]), w_g);
    a2 = vmlal_u8(a2, vget_low_u8(p1.val[2]), w_b);
    uint16x8_t a3 = vmull_u8(vget_high_u8(p1.val[0]), w_r);
    a3 = vmlal_u8(a3, vget_high_u8(p1.val[1]), w_g);
    a3 = vmlal_u8(a3, vget_high_u8(p1.val[2]), w_b);

    vst1q_u8(gray + i, vcombine_u8(vrshrn_n_u16(a0, 8), vrshrn_n_u16(a1, 8)));
    vst1q_u8(gray + i + 16,
             vcombine_u8(vrshrn_n_u16(a2, 8), vrshrn_n_u16(a3, 8)));
  }
  for (; i <= n - 16; i += 16) {
    uint8x16x3_t pix = vld3q_u8(rgb + i * 3);
    uint16x8_t lo = vmull_u8(vget_low_u8(pix.val[0]), w_r);
    lo = vmlal_u8(lo, vget_low_u8(pix.val[1]), w_g);
    lo = vmlal_u8(lo, vget_low_u8(pix.val[2]), w_b);
    uint16x8_t hi = vmull_u8(vget_high_u8(pix.val[0]), w_r);
    hi = vmlal_u8(hi, vget_high_u8(pix.val[1]), w_g);
    hi = vmlal_u8(hi, vget_high_u8(pix.val[2]), w_b);
    vst1q_u8(gray + i, vcombine_u8(vrshrn_n_u16(lo, 8), vrshrn_n_u16(hi, 8)));
  }
  for (; i < n; i++) {
    uint32_t r = rgb[3 * i + 0], g = rgb[3 * i + 1], b = rgb[3 * i + 2];
    gray[i] = (uint8_t)((WR * r + WG * g + WB * b + 128) >> 8);
  }
}

// Benchmark helper: run kernel NTIMES and return the AVERAGE time (ms)
typedef void (*gray_fn)(const uint8_t*, uint8_t*, long);

static double bench(gray_fn fn, const uint8_t* rgb, uint8_t* gray, long n) {
  double start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) fn(rgb, gray, n);
  double end = get_time_ms();
  return (end - start) / NTIMES;
}

int main(int argc, char* argv[]) {
  long n = (argc == 2) ? atol(argv[1]) : (1920 * 1080);  // default 1080p

  printf("=======================================================\n");
  printf(" RGB2Gray v3: Add NEON Loop-Unrolled (BT.601)\n");
  printf(" Gray = (77*R + 150*G + 29*B + 128) >> 8\n");
  printf(" Pixels: %ld   Loops: %d\n", n, NTIMES);
  printf("=======================================================\n");

  // Aligned allocation (16-byte) for SIMD efficiency
  size_t bytes_rgb = ((size_t)n * 3 + 15) & ~(size_t)15;
  size_t bytes_g   = ((size_t)n     + 15) & ~(size_t)15;
  uint8_t* rgb = (uint8_t*)aligned_alloc(16, bytes_rgb);
  uint8_t* g_ref = (uint8_t*)aligned_alloc(16, bytes_g);
  uint8_t* g_opt = (uint8_t*)aligned_alloc(16, bytes_g);
  if (!rgb || !g_ref || !g_opt) {
    printf("Alloc failed\n");
    return 1;
  }

  for (long i = 0; i < n; i++) {
    rgb[3 * i + 0] = (uint8_t)((i * 7) & 0xFF);
    rgb[3 * i + 1] = (uint8_t)((i * 13) & 0xFF);
    rgb[3 * i + 2] = (uint8_t)((i * 29) & 0xFF);
  }

  // Golden reference = Serial (No-Vec)
  rgb2gray_serial_no_vec(rgb, g_ref, n);

  memset(g_opt, 0, bytes_g);
  double t_nv = bench(rgb2gray_serial_no_vec, rgb, g_opt, n);
  memset(g_opt, 0, bytes_g);
  double t_se = bench(rgb2gray_serial, rgb, g_opt, n);
  const char* s_se = check_diff(g_ref, g_opt, n);
  memset(g_opt, 0, bytes_g);
  double t_ne = bench(rgb2gray_neon, rgb, g_opt, n);
  const char* s_ne = check_diff(g_ref, g_opt, n);
  memset(g_opt, 0, bytes_g);
  double t_un = bench(rgb2gray_neon_unroll, rgb, g_opt, n);
  const char* s_un = check_diff(g_ref, g_opt, n);

  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |   -   |\n", t_nv);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_se, t_nv / t_se,
         s_se);
  printf("| NEON Intrinsic  | %9.3f | %5.2f x |  %-4s |\n", t_ne, t_nv / t_ne,
         s_ne);
  printf("| NEON Unrolled   | %9.3f | %5.2f x |  %-4s |\n", t_un, t_nv / t_un,
         s_un);
  printf("-------------------------------------------------\n");

  free(rgb);
  free(g_ref);
  free(g_opt);
  return 0;
}

In [ ]:
BIN = compile_c("src_rgb2gray/rgb2gray_v3.c", "src_rgb2gray/rgb2gray_v3")
out_v3 = run_bin(BIN, 2073600)

## 8. 📈 性能可视化（基于 v3 的四版本结果）

v3 的输出包含全部四个实现的耗时与加速比，据此绘制柱状图，以完整呈现各手段的效果。
（灰色表示无明显加速，蓝色表示存在加速，**红色标示性能最优的版本**）

In [ ]:
rows_v3 = parse_table(out_v3)
for r in rows_v3:
    print(f'{r["method"]:16s} {r["time"]:9.3f} ms   {r["speedup"]:.2f}x')
plot_speedup(rows_v3, "RGB to Gray: performance of four implementations (2.07M px)")

## 9. 结果分析

> 注：具体数值随硬件平台、图像尺寸、编译器版本与系统负载而变化，请以本机实际运行结果为准；下述分析针对数据所反映的**趋势与规律**。

RGB→Gray 的结果通常呈现以下两个值得注意的现象：

**① 加速比明显高于 AXPY。**

相比实验一 AXPY，本实验的算术强度更高（读取 3 字节、写入 1 字节，却执行多次乘加），且数据为 u8，SIMD 的并行度更大（一个 128 位向量可容纳更多元素）。因此其加速比通常更高。

**② 自动向量化与手写 NEON 性能几乎相同——甚至前者略快。**

RGB→Gray 是**规整的逐像素运算**，不含跨迭代依赖，编译器能够对其有效地自动向量化。这不只是"性能接近"，而是编译器**自己生成了同一套向量指令**。用 `gcc -O3 -S` 观察 `rgb2gray_serial` 的主循环：

```asm
.L13:
  ld3    {v4.16b - v6.16b}, [x4], 48   // 解交织 16 个像素 —— 等价于 vld3q_u8
  umull  v1.8h, v5.8b,  v7.8b          // 0x96 = 150 → G（低 8 路）
  umull2 v0.8h, v5.16b, v7.16b         //                （高 8 路）
  umlal  v1.8h, v4.8b,  v16.8b         // 0x4d = 77  → R
  umlal2 v0.8h, v4.16b, v16.16b
  add    v1.8h, v1.8h, v3.8h           // +128（v3 = 0x80）
  add    v0.8h, v0.8h, v3.8h
  umlal  v1.8h, v6.8b,  v2.8b          // 0x1d = 29  → B
  umlal2 v0.8h, v6.16b, v2.16b
  ushr   v1.8h, v1.8h, 8               // >>8
  ushr   v0.8h, v0.8h, 8
  xtn    v4.8b,  v1.8h                 // 窄化回 u8
  xtn2   v4.16b, v0.8h
  str    q4, [x3], 16                  // 一次写回 16 个灰度像素
  cmp    x3, x5
  bne    .L13
```

解交织 → 加宽乘加 → 加 128 → 右移 8 → 窄化 → 写回，全部由编译器自己完成，与手写版本一一对应。

> 💡 **不同 GCC 版本的写法差异**：末尾的 `ushr` + `xtn`/`xtn2` 与手写版的 `vrshrn_n_u16` 等价。GCC 12 及以后会把这三条合成一条 `uzp2`（直接抽取每个 16 位元素的高字节，即 `>>8` 后窄化）。

**③ 三个向量化版本耗时几乎相同 → 瓶颈已从计算转到访存。**

每调用一次核函数需搬运 $4n$ 字节（读 $3n$ + 写 $n$），$n = 2073600$ 时约 8.3 MB。以本次记录的耗时估算有效带宽：

| 版本 | 耗时 (ms) | 有效带宽 (GB/s) |
|---|---|---|
| Serial (No-Vec) | 1.808 | ≈ 4.6 |
| Serial (Auto) | 0.335 | ≈ 24.8 |
| NEON Intrinsic | 0.346 | ≈ 24.0 |
| NEON Unrolled | 0.341 | ≈ 24.3 |

三个向量化版本都停在同一条"带宽墙"上（≈24–25 GB/s）。这也解释了为何 SIMD 理论并行度是 16 路，实测加速比却只有 5.x——ALU 已不再是瓶颈，再怎么优化计算也无济于事。

> **规律：编译器能否有效自动向量化，取决于三件事——(a) 是否存在它不敢打破的依赖；(b) 目标架构是否有对应的指令模式；(c) 成本模型是否认为划算。**
> - **浮点规约**（实验二 GEMV）：受结合律限制，编译器不敢重排累加顺序（除非 `-ffast-math`），属于情形 (a) → 必须手写。
> - **纯重排**（实验三 RGB→BGR）：注意**不能**说"stride-3 交织访存无法向量化"——本实验的 `Serial (Auto)` 恰恰生成了 `ld3`，说明情形 (b) 是满足的。RGB→BGR 未被自动向量化，是因为它是**原地读-改-写的通道交换**，循环体内没有任何算术运算，编译器成本模型认为向量化"不划算"（用 `-fopt-info-vec-missed` 可看到 *Loop costings may not be worthwhile*），属于情形 (c)。改成非原地写法后 GCC 就会生成 `ld3`/`st3`。
> - **规约统计 + 饱和**（实验四 白平衡）：整数加法满足结合律，编译器可以安全重排累加顺序，因此自动向量化**同样有效**——实验四实测 `Serial (Auto)` 和手写 `NEON` 二者基本打平，真正的提升来自循环展开。可见"含规约"并不等于"必须手写"。
> - **规整逐像素运算**（本实验）：三者都不成问题，编译器可完全胜任。

---

### 🎓 结论
RGB→Gray 展示了**跨通道加权规约**与**定点加宽乘加**的实现方法，并给出一个重要判断依据：**对于规整的、访存受限的逐像素负载，启用 `-O3` 让编译器自动向量化通常已经足够，手写 NEON 不会更快。** 手写 NEON 的价值，主要体现在编译器因依赖、指令模式或成本模型而放弃优化的运算上——先测量、看汇编、再决定是否手写。

## 10. 🔧 动手练习

请修改代码、重新编译并运行，观察性能的变化（建议先独立完成，再阅读思考题）：

1. 将权重改为三通道简单平均（如 85 / 86 / 85，和约为 256），观察数值差异。
2. 仅保留 `Serial (Auto)` 与 `NEON Intrinsic` 两个版本，多次运行取平均，验证二者性能是否确实接近。
3. 将输出也改为三通道灰度（用 `vst3` 写回），观察写回交织是否会拖慢速度。
4. 【进阶】为 `compile_c` 的编译参数添加 `-march=native` 后重新运行，观察 `Serial (Auto)` 是否进一步提升。

## 11. 🤔 思考题

- 为何 RGB→Gray 的自动向量化即可获得良好性能，而实验二 GEMV 却必须手写 NEON？
- 为何本实验的加速比明显高于实验一 AXPY？（可从算术强度与数据位宽两方面分析）
- 定点权重为何要凑成和为 256？若权重之和为 255 或 257 会有何影响？
- 加宽乘加中，为何必须先将 u8 加宽到 u16 再累加？若不加宽会发生什么？
- 四个版本中三个向量化版本耗时几乎相同，这说明瓶颈在哪里？

## 12. 小结与后续

本实验完成了 RGB→Gray 从串行到 NEON 的实现过程：

<table>
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">新增内容</th>
      <th style="text-align: left;">涉及知识点</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>v1</strong></td>
      <td style="text-align: left;"><code>rgb2gray_serial_no_vec</code> + <code>rgb2gray_serial</code></td>
      <td style="text-align: left;">性能基准、定点化、编译器自动向量化</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v2</strong></td>
      <td style="text-align: left;"><code>rgb2gray_neon</code></td>
      <td style="text-align: left;">跨通道加权规约、加宽乘加、舍入窄化</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v3</strong></td>
      <td style="text-align: left;"><code>rgb2gray_neon_unroll</code></td>
      <td style="text-align: left;">循环展开</td>
    </tr>
  </tbody>
</table>

RGB→Gray 展示了**跨通道加权规约**与**定点加宽乘加**，并给出一条重要规律：**对规整的逐像素负载，编译器自动向量化往往已足够，手写 NEON 的价值体现在编译器难以处理的运算上。**

➡️ **后续内容：实验六 YUV → RGB**。作为本章的综合案例，它将把**色度上采样、类型提升、定点运算、饱和窄化与无分支处理**整合于一条完整的视频解码流程之中。